In [2]:
# 1. IMPORT LIBRARY DAN PENGATURAN FOLDER
# Jika library belum tersedia, install melalui terminal:
# python -m pip install pandas numpy

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

print("Library berhasil diimpor.")
print("Folder kerja:", Path.cwd())

Library berhasil diimpor.
Folder kerja: C:\Users\Carxiel\Documents\Tugas Kuliah\Machine Learning\Tugas Binary, Multiclass, Multilabel


In [3]:
# 2. MENEMUKAN DAN MEMBACA SEMBILAN DATASET

nama_file = {
    "heart_failure": "heart_failure_clinical_records_dataset.csv",
    "mbg_indonesia": "indonesia_mbg_nationwide_enriched_2026.csv",
    "kepler": "kepler_koi_cumulative.csv",
    "mall_customers": "Mall_Customers.csv",
    "pokemon": "pokemon_alopez247.csv",
    "steam": "steam.csv",
    "titanic": "Titanic-Dataset.csv",
    "ufo_sightings": "ufo-sightings.csv",
    "wine_quality": "WineQT.csv"
}

# Cari folder yang berisi file CSV.
DATA_DIR = Path.cwd()

if not all((DATA_DIR / nama).is_file() for nama in nama_file.values()):
    kandidat = list(DATA_DIR.rglob(nama_file["heart_failure"]))
    if kandidat:
        DATA_DIR = kandidat[0].parent

file_tidak_ada = [
    nama for nama in nama_file.values()
    if not (DATA_DIR / nama).is_file()
]

if file_tidak_ada:
    raise FileNotFoundError(
        "File berikut belum ditemukan:\n"
        + "\n".join(file_tidak_ada)
        + "\n\nSimpan notebook dan sembilan CSV dalam folder yang sama."
    )

OUTPUT_DIR = DATA_DIR / "hasil_klasifikasi"
(OUTPUT_DIR / "binary").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "multiclass").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "multilabel").mkdir(parents=True, exist_ok=True)

def baca_csv(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1", low_memory=False)

datasets = {
    key: baca_csv(DATA_DIR / nama)
    for key, nama in nama_file.items()
}

print("Berhasil membaca", len(datasets), "dataset.")
print("Lokasi dataset:", DATA_DIR)
print("Lokasi hasil:", OUTPUT_DIR)

Berhasil membaca 9 dataset.
Lokasi dataset: C:\Users\Carxiel\Documents\Tugas Kuliah\Machine Learning\Tugas Binary, Multiclass, Multilabel
Lokasi hasil: C:\Users\Carxiel\Documents\Tugas Kuliah\Machine Learning\Tugas Binary, Multiclass, Multilabel\hasil_klasifikasi


In [4]:
# 3. MELIHAT RINGKASAN SEMBILAN DATASET

ringkasan_awal = []

for nama, df in datasets.items():
    ringkasan_awal.append({
        "Dataset": nama,
        "Jumlah Baris": len(df),
        "Jumlah Kolom": len(df.columns),
        "Missing Values": int(df.isna().sum().sum())
    })

df_ringkasan_awal = pd.DataFrame(ringkasan_awal)
display(df_ringkasan_awal)

print("Total seluruh baris:", df_ringkasan_awal["Jumlah Baris"].sum())

,Dataset,Jumlah Baris,Jumlah Kolom,Missing Values
0,heart_failure,299,13,0
1,mbg_indonesia,13000,26,0
2,kepler,9564,50,40557
3,mall_customers,200,5,0
4,pokemon,721,23,978
5,steam,27075,18,15
6,titanic,891,12,866
7,ufo_sightings,88678,11,25752
8,wine_quality,1143,13,0


Total seluruh baris: 141571


In [5]:
# 4. FUNGSI UNTUK MENYIMPAN HASIL KLASIFIKASI

ringkasan_hasil = []
panduan_label = []

def slugify(teks):
    teks = str(teks).strip().lower()
    teks = re.sub(r"[^a-z0-9]+", "_", teks)
    return teks.strip("_")

def simpan_hasil(
    nama_dataset,
    data_sumber,
    target,
    jenis,
    kolom_dihapus,
    keterangan
):
    # Pastikan target selalu berbentuk DataFrame.
    if isinstance(target, pd.Series):
        target = target.to_frame()

    target = target.copy().reset_index(drop=True)
    fitur = data_sumber.drop(
        columns=list(kolom_dihapus),
        errors="ignore"
    ).copy().reset_index(drop=True)

    # Hilangkan kolom fitur yang seluruh nilainya kosong.
    kolom_kosong = [
        kolom for kolom in fitur.columns
        if fitur[kolom].isna().all()
    ]
    if kolom_kosong:
        fitur = fitur.drop(columns=kolom_kosong)

    # Gabungkan fitur dengan target.
    hasil = pd.concat([fitur, target], axis=1)

    # Pemeriksaan dasar sebelum file disimpan.
    if len(hasil) != len(data_sumber):
        raise ValueError(f"Jumlah baris berubah pada {nama_dataset} - {jenis}.")

    if target.isna().any().any():
        raise ValueError(f"Target memiliki nilai kosong pada {nama_dataset} - {jenis}.")

    if jenis in ["binary", "multilabel"]:
        for kolom in target.columns:
            nilai = set(pd.Series(target[kolom]).dropna().unique())
            if not nilai.issubset({0, 1, False, True}):
                raise ValueError(f"Target {kolom} bukan bernilai 0/1.")

    folder = OUTPUT_DIR / jenis
    folder.mkdir(parents=True, exist_ok=True)
    nama_output = f"{nama_dataset}_{jenis}.csv"
    lokasi_output = folder / nama_output
    hasil.to_csv(lokasi_output, index=False)

    # Membuat informasi distribusi label.
    if jenis == "multilabel":
        distribusi = {
            kolom: int(target[kolom].sum())
            for kolom in target.columns
        }
    else:
        distribusi = {
            str(label): int(jumlah)
            for label, jumlah in target.iloc[:, 0].value_counts().items()
        }

    ringkasan_hasil.append({
        "Dataset": nama_dataset,
        "Jenis Klasifikasi": jenis,
        "Jumlah Baris": len(hasil),
        "Jumlah Fitur": len(fitur.columns),
        "Kolom Target": ", ".join(target.columns),
        "Distribusi Label": json.dumps(distribusi, ensure_ascii=False),
        "File Output": str(lokasi_output.relative_to(DATA_DIR))
    })

    panduan_label.append({
        "Dataset": nama_dataset,
        "Jenis Klasifikasi": jenis,
        "Kolom Target": ", ".join(target.columns),
        "Kolom Sumber Target Dihapus": ", ".join(kolom_dihapus),
        "Keterangan Label": keterangan
    })

    print(
        f"{nama_dataset} | {jenis} | "
        f"{len(hasil)} baris | tersimpan: {lokasi_output.name}"
    )
    return hasil

print("Fungsi penyimpanan siap digunakan.")

Fungsi penyimpanan siap digunakan.


In [6]:
# 5. DATASET HEART FAILURE

df = datasets["heart_failure"]

# BINARY: 0 = tidak meninggal selama follow-up, 1 = meninggal.
target_binary = df["DEATH_EVENT"].astype(int).rename("target_binary")
hasil_hf_binary = simpan_hasil(
    "heart_failure", df, target_binary, "binary",
    ["DEATH_EVENT"],
    "Menggunakan label asli DEATH_EVENT: 0 tidak meninggal selama follow-up, 1 meninggal."
)

# MULTICLASS: kelompok umur.
umur = pd.to_numeric(df["age"], errors="coerce")
target_multiclass = pd.cut(
    umur,
    bins=[-np.inf, 50, 65, np.inf],
    labels=["Under 50", "50-64", "65+"],
    right=False
).astype("string").rename("target_multiclass")

hasil_hf_multiclass = simpan_hasil(
    "heart_failure", df, target_multiclass, "multiclass",
    ["age", "DEATH_EVENT"],
    "Kelompok umur turunan: Under 50, 50-64, dan 65+ tahun."
)

# MULTILABEL: beberapa kondisi dapat dimiliki pasien yang sama.
target_multilabel = df[
    ["anaemia", "diabetes", "high_blood_pressure", "smoking"]
].astype(int).rename(columns={
    "anaemia": "label_anaemia",
    "diabetes": "label_diabetes",
    "high_blood_pressure": "label_high_blood_pressure",
    "smoking": "label_smoking"
})

hasil_hf_multilabel = simpan_hasil(
    "heart_failure", df, target_multilabel, "multilabel",
    ["anaemia", "diabetes", "high_blood_pressure", "smoking", "DEATH_EVENT"],
    "Empat indikator biner dari kolom asli: anemia, diabetes, tekanan darah tinggi, dan merokok."
)

heart_failure | binary | 299 baris | tersimpan: heart_failure_binary.csv
heart_failure | multiclass | 299 baris | tersimpan: heart_failure_multiclass.csv
heart_failure | multilabel | 299 baris | tersimpan: heart_failure_multilabel.csv


In [7]:
# 6. DATASET INDONESIA MBG

df = datasets["mbg_indonesia"]

# BINARY: tier yang compliant menjadi 1, tier lain menjadi 0.
tier = df["bgn_program_compliance_tier"].astype("string").str.strip()
tier_patuh = ["Standard-Compliant", "Exemplary-Gold-Standard"]
target_binary = tier.isin(tier_patuh).astype(int).rename("target_binary")

hasil_mbg_binary = simpan_hasil(
    "mbg_indonesia", df, target_binary, "binary",
    ["distribution_id", "bgn_program_compliance_tier"],
    "1 untuk Standard-Compliant atau Exemplary-Gold-Standard; 0 untuk tier lainnya."
)

# MULTICLASS: menggunakan empat kategori compliance asli.
target_multiclass = tier.rename("target_multiclass")

hasil_mbg_multiclass = simpan_hasil(
    "mbg_indonesia", df, target_multiclass, "multiclass",
    ["distribution_id", "bgn_program_compliance_tier"],
    "Empat kategori asli bgn_program_compliance_tier."
)

# MULTILABEL: indikator risiko turunan dari nilai kuartil dataset.
q75_waste = df["plate_waste_percentage"].quantile(0.75)
q75_spoilage = df["spoilage_risk_index"].quantile(0.75)
q75_distance = df["delivery_distance_km"].quantile(0.75)
q25_attendance = df["student_attendance_rate_pct"].quantile(0.25)
q25_local = df["local_farmer_sourcing_pct"].quantile(0.25)

target_multilabel = pd.DataFrame({
    "label_high_plate_waste": (df["plate_waste_percentage"] >= q75_waste).astype(int),
    "label_high_spoilage_risk": (df["spoilage_risk_index"] >= q75_spoilage).astype(int),
    "label_long_delivery_distance": (df["delivery_distance_km"] >= q75_distance).astype(int),
    "label_low_attendance": (df["student_attendance_rate_pct"] <= q25_attendance).astype(int),
    "label_low_local_sourcing": (df["local_farmer_sourcing_pct"] <= q25_local).astype(int)
})

kolom_sumber_label = [
    "plate_waste_percentage",
    "spoilage_risk_index",
    "delivery_distance_km",
    "student_attendance_rate_pct",
    "local_farmer_sourcing_pct"
]

hasil_mbg_multilabel = simpan_hasil(
    "mbg_indonesia", df, target_multilabel, "multilabel",
    ["distribution_id", "bgn_program_compliance_tier",
     "primary_food_waste_driver", "cot_nutrition_logistics_rationale"]
    + kolom_sumber_label,
    "Lima indikator risiko turunan dari kuartil: limbah makanan tinggi, risiko pembusukan tinggi, jarak jauh, kehadiran rendah, dan sumber petani lokal rendah. Ambang disimpan di output notebook."
)

print("Ambang kuartil MBG:")
print({
    "limbah makanan Q75": round(q75_waste, 3),
    "risiko pembusukan Q75": round(q75_spoilage, 3),
    "jarak pengiriman Q75": round(q75_distance, 3),
    "kehadiran Q25": round(q25_attendance, 3),
    "sumber petani lokal Q25": round(q25_local, 3)
})

mbg_indonesia | binary | 13000 baris | tersimpan: mbg_indonesia_binary.csv
mbg_indonesia | multiclass | 13000 baris | tersimpan: mbg_indonesia_multiclass.csv
mbg_indonesia | multilabel | 13000 baris | tersimpan: mbg_indonesia_multilabel.csv
Ambang kuartil MBG:
{'limbah makanan Q75': np.float64(17.0), 'risiko pembusukan Q75': np.float64(2.8), 'jarak pengiriman Q75': np.float64(9.1), 'kehadiran Q25': np.float64(92.1), 'sumber petani lokal Q25': np.float64(65.8)}


In [8]:
# 7. DATASET KEPLER EXOPLANET

df = datasets["kepler"]
disposition = df["koi_disposition"].astype("string").str.strip().str.upper()

kolom_id_kepler = ["rowid", "kepid", "kepoi_name", "kepler_name"]

# BINARY: 1 = CONFIRMED, 0 = selain CONFIRMED.
target_binary = disposition.eq("CONFIRMED").astype(int).rename("target_binary")

hasil_kepler_binary = simpan_hasil(
    "kepler", df, target_binary, "binary",
    kolom_id_kepler + ["koi_disposition", "koi_pdisposition"],
    "1 untuk CONFIRMED dan 0 untuk CANDIDATE atau FALSE POSITIVE."
)

# MULTICLASS: tiga disposition.
target_multiclass = disposition.rename("target_multiclass")

hasil_kepler_multiclass = simpan_hasil(
    "kepler", df, target_multiclass, "multiclass",
    kolom_id_kepler + ["koi_disposition", "koi_pdisposition"],
    "Menggunakan tiga kelas koi_disposition: CONFIRMED, CANDIDATE, dan FALSE POSITIVE."
)

# MULTILABEL: empat false-positive flags asli.
kolom_flag = ["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
target_multilabel = df[kolom_flag].fillna(0).astype(int).rename(columns={
    "koi_fpflag_nt": "label_fp_nt",
    "koi_fpflag_ss": "label_fp_ss",
    "koi_fpflag_co": "label_fp_co",
    "koi_fpflag_ec": "label_fp_ec"
})

hasil_kepler_multilabel = simpan_hasil(
    "kepler", df, target_multilabel, "multilabel",
    kolom_id_kepler + kolom_flag + ["koi_disposition", "koi_pdisposition"],
    "Empat indikator false-positive flags dari kolom asli Kepler."
)

kepler | binary | 9564 baris | tersimpan: kepler_binary.csv
kepler | multiclass | 9564 baris | tersimpan: kepler_multiclass.csv
kepler | multilabel | 9564 baris | tersimpan: kepler_multilabel.csv


In [9]:
# 8. DATASET MALL CUSTOMERS

df = datasets["mall_customers"]
spending = pd.to_numeric(df["Spending Score (1-100)"], errors="coerce")
income = pd.to_numeric(df["Annual Income (k$)"], errors="coerce")
age = pd.to_numeric(df["Age"], errors="coerce")

# BINARY: spending score >= 60 dianggap high spender.
target_binary = spending.ge(60).astype(int).rename("target_binary")

hasil_mall_binary = simpan_hasil(
    "mall_customers", df, target_binary, "binary",
    ["CustomerID", "Spending Score (1-100)"],
    "1 jika Spending Score >= 60, selain itu 0. Ambang 60 adalah aturan untuk latihan."
)

# MULTICLASS: low, medium, high spending.
target_multiclass = pd.cut(
    spending,
    bins=[-np.inf, 40, 60, np.inf],
    labels=["Low (<=40)", "Medium (41-60)", "High (>60)"],
    right=True
).astype("string").rename("target_multiclass")

hasil_mall_multiclass = simpan_hasil(
    "mall_customers", df, target_multiclass, "multiclass",
    ["CustomerID", "Spending Score (1-100)"],
    "Kategori spending score: Low <=40, Medium 41-60, High >60."
)

# MULTILABEL: beberapa karakteristik pelanggan dapat muncul bersamaan.
median_income = income.median()
target_multilabel = pd.DataFrame({
    "label_high_income": income.ge(median_income).astype(int),
    "label_high_spender": spending.ge(60).astype(int)
})

hasil_mall_multilabel = simpan_hasil(
    "mall_customers", df, target_multilabel, "multilabel",
    ["CustomerID", "Annual Income (k$)", "Spending Score (1-100)"],
    f"Dua tag turunan pelanggan: pendapatan tinggi (>= median {median_income:g}) dan high spender (score >=60). Gender dan umur dipertahankan sebagai fitur."
)

mall_customers | binary | 200 baris | tersimpan: mall_customers_binary.csv
mall_customers | multiclass | 200 baris | tersimpan: mall_customers_multiclass.csv
mall_customers | multilabel | 200 baris | tersimpan: mall_customers_multilabel.csv


In [10]:
# 9. DATASET POKEMON

df = datasets["pokemon"]

# BINARY: Pokemon legendary atau bukan.
target_binary = df["isLegendary"].astype(int).rename("target_binary")

hasil_pokemon_binary = simpan_hasil(
    "pokemon", df, target_binary, "binary",
    ["Number", "Name", "isLegendary"],
    "Menggunakan label asli isLegendary: 1 legendary, 0 bukan legendary."
)

# MULTICLASS: Type_1 sebagai tipe utama.
target_multiclass = (
    df["Type_1"].astype("string").str.strip()
    .fillna("Unknown").rename("target_multiclass")
)

hasil_pokemon_multiclass = simpan_hasil(
    "pokemon", df, target_multiclass, "multiclass",
    ["Number", "Name", "Type_1"],
    "Menggunakan Type_1 sebagai kelas tipe utama Pokemon."
)

# MULTILABEL: kombinasi Type_1 dan Type_2 dibuat menjadi multi-hot labels.
tipe_1 = df["Type_1"].astype("string").str.strip()
tipe_2 = df["Type_2"].astype("string").str.strip()
semua_tipe = sorted(set(tipe_1.dropna()) | set(tipe_2.dropna()))

target_multilabel = pd.DataFrame(index=df.index)
for tipe in semua_tipe:
    nama_label = "label_type_" + slugify(tipe)
    target_multilabel[nama_label] = (
        tipe_1.eq(tipe).fillna(False) | tipe_2.eq(tipe).fillna(False)
    ).astype(int)

hasil_pokemon_multilabel = simpan_hasil(
    "pokemon", df, target_multilabel, "multilabel",
    ["Number", "Name", "Type_1", "Type_2"],
    "One-hot multi-label dari seluruh tipe yang muncul pada Type_1 atau Type_2."
)

print("Jumlah label tipe Pokemon:", len(semua_tipe))

pokemon | binary | 721 baris | tersimpan: pokemon_binary.csv
pokemon | multiclass | 721 baris | tersimpan: pokemon_multiclass.csv
pokemon | multilabel | 721 baris | tersimpan: pokemon_multilabel.csv
Jumlah label tipe Pokemon: 18


In [11]:
# 10. DATASET STEAM GAMES

df = datasets["steam"]

harga = pd.to_numeric(df["price"], errors="coerce")

# BINARY: game gratis atau berbayar.
target_binary = harga.eq(0).astype(int).rename("target_binary")

hasil_steam_binary = simpan_hasil(
    "steam", df, target_binary, "binary",
    ["appid", "name", "price"],
    "1 jika harga game 0 (gratis), 0 jika harga lebih dari 0 (berbayar)."
)

# MULTICLASS: genre pertama pada kolom genres.
genre_text = df["genres"].fillna("Unknown").astype(str)
genre_utama = genre_text.str.split(";").str[0].str.strip().replace("", "Unknown")
target_multiclass = genre_utama.rename("target_multiclass")

hasil_steam_multiclass = simpan_hasil(
    "steam", df, target_multiclass, "multiclass",
    ["appid", "name", "genres", "steamspy_tags"],
    "Menggunakan genre pertama pada kolom genres sebagai genre utama."
)

# MULTILABEL: semua genre dalam satu baris diubah menjadi beberapa indikator 0/1.
daftar_genre = genre_text.apply(
    lambda teks: [item.strip() for item in teks.split(";") if item.strip()]
)
semua_genre = sorted({genre for daftar in daftar_genre for genre in daftar})

target_multilabel = pd.DataFrame(index=df.index)
for genre in semua_genre:
    nama_label = "label_genre_" + slugify(genre)
    target_multilabel[nama_label] = daftar_genre.apply(
        lambda daftar: int(genre in daftar)
    )

hasil_steam_multilabel = simpan_hasil(
    "steam", df, target_multilabel, "multilabel",
    ["appid", "name", "genres", "steamspy_tags"],
    "Multi-hot encoding dari seluruh genre Steam yang dipisahkan dengan tanda titik koma."
)

print("Jumlah genre Steam:", len(semua_genre))

steam | binary | 27075 baris | tersimpan: steam_binary.csv
steam | multiclass | 27075 baris | tersimpan: steam_multiclass.csv
steam | multilabel | 27075 baris | tersimpan: steam_multilabel.csv
Jumlah genre Steam: 29


In [12]:
# 11. DATASET TITANIC

df = datasets["titanic"]

# BINARY: survived (1) atau tidak survived (0).
target_binary = df["Survived"].astype(int).rename("target_binary")

hasil_titanic_binary = simpan_hasil(
    "titanic", df, target_binary, "binary",
    ["PassengerId", "Name", "Ticket", "Survived"],
    "Menggunakan label asli Survived: 1 selamat, 0 tidak selamat."
)

# MULTICLASS: kelas tiket penumpang.
target_multiclass = (
    "Class " + df["Pclass"].astype(str)
).rename("target_multiclass")

hasil_titanic_multiclass = simpan_hasil(
    "titanic", df, target_multiclass, "multiclass",
    ["PassengerId", "Name", "Ticket", "Pclass", "Survived"],
    "Menggunakan Pclass sebagai tiga kelas tiket: Class 1, Class 2, dan Class 3."
)

# MULTILABEL: tag karakteristik penumpang.
umur = pd.to_numeric(df["Age"], errors="coerce")
keluarga = pd.to_numeric(df["SibSp"], errors="coerce").fillna(0) + pd.to_numeric(
    df["Parch"], errors="coerce"
).fillna(0)

target_multilabel = pd.DataFrame({
    "label_female": df["Sex"].astype("string").str.lower().eq("female").astype(int),
    "label_child": umur.lt(18).fillna(False).astype(int),
    "label_traveling_with_family": keluarga.gt(0).astype(int),
    "label_cabin_recorded": df["Cabin"].notna().astype(int),
    "label_embarked_southampton": df["Embarked"].astype("string").eq("S").fillna(False).astype(int),
    "label_age_unknown": umur.isna().astype(int)
})

hasil_titanic_multilabel = simpan_hasil(
    "titanic", df, target_multilabel, "multilabel",
    ["PassengerId", "Name", "Ticket", "Survived", "Sex", "Age",
     "SibSp", "Parch", "Cabin", "Embarked"],
    "Enam tag karakteristik: female, child (<18), bersama keluarga, cabin tercatat, embarkasi Southampton, dan umur tidak diketahui."
)

titanic | binary | 891 baris | tersimpan: titanic_binary.csv
titanic | multiclass | 891 baris | tersimpan: titanic_multiclass.csv
titanic | multilabel | 891 baris | tersimpan: titanic_multilabel.csv


In [13]:
# 12. DATASET UFO SIGHTINGS

df = datasets["ufo_sightings"].copy()

shape = (
    df["shape"].astype("string").str.strip().str.lower()
    .replace("", pd.NA).fillna("unknown")
)

# BINARY: bentuk objek dapat diidentifikasi atau tidak.
target_binary = (~shape.isin(["unknown", "other"])).astype(int).rename("target_binary")

hasil_ufo_binary = simpan_hasil(
    "ufo_sightings", df, target_binary, "binary",
    ["shape"],
    "1 jika shape tercatat sebagai bentuk selain unknown/other; 0 jika unknown, other, atau kosong."
)

# MULTICLASS: menggunakan kategori shape.
target_multiclass = shape.rename("target_multiclass")

hasil_ufo_multiclass = simpan_hasil(
    "ufo_sightings", df, target_multiclass, "multiclass",
    ["shape"],
    "Menggunakan kategori bentuk UFO pada kolom shape; nilai kosong menjadi unknown."
)

# MULTILABEL: tag dari deskripsi laporan. Ini aturan kata kunci, bukan label asli.
komentar = df["comments"].fillna("").astype(str).str.lower()

pola_label = {
    "label_mentions_light": r"\b(?:light|lights|bright|glowing|glow|illumina(?:ted|tion)|flash|flashing|orb|orbs)\b",
    "label_mentions_movement": r"\b(?:move|moving|moved|flew|flying|hover|hovering|travel|travelling|drift|drifting|zigzag|maneuver|speed)\b",
    "label_mentions_multiple_objects": r"\b(?:formation|several|multiple|many|group|fleet|two|three|four|five|objects)\b",
    "label_mentions_color": r"\b(?:red|blue|green|white|orange|yellow|purple|silver|golden)\b",
    "label_mentions_sound": r"\b(?:sound|noise|humming|hum|buzz|buzzing|roar|rumble|bang|crackling)\b"
}

target_multilabel = pd.DataFrame({
    nama_label: komentar.str.contains(pola, regex=True, na=False).astype(int)
    for nama_label, pola in pola_label.items()
})

hasil_ufo_multilabel = simpan_hasil(
    "ufo_sightings", df, target_multilabel, "multilabel",
    ["comments"],
    "Lima tag heuristik berdasarkan kata kunci deskripsi: cahaya, gerakan, banyak objek, warna, dan suara. Label ini perlu ditafsirkan sebagai hasil pencarian kata, bukan verifikasi kejadian."
)

ufo_sightings | binary | 88678 baris | tersimpan: ufo_sightings_binary.csv
ufo_sightings | multiclass | 88678 baris | tersimpan: ufo_sightings_multiclass.csv
ufo_sightings | multilabel | 88678 baris | tersimpan: ufo_sightings_multilabel.csv


In [14]:
# 13. DATASET WINE QUALITY

df = datasets["wine_quality"]
quality = pd.to_numeric(df["quality"], errors="coerce")

# BINARY: kualitas baik jika quality >= 7.
target_binary = quality.ge(7).astype(int).rename("target_binary")

hasil_wine_binary = simpan_hasil(
    "wine_quality", df, target_binary, "binary",
    ["Id", "quality"],
    "1 jika quality >= 7 (baik), 0 jika quality < 7."
)

# MULTICLASS: kualitas rendah, sedang, dan tinggi.
target_multiclass = pd.cut(
    quality,
    bins=[-np.inf, 5, 6, np.inf],
    labels=["Low (<=5)", "Medium (6)", "High (>=7)"],
    right=True
).astype("string").rename("target_multiclass")

hasil_wine_multiclass = simpan_hasil(
    "wine_quality", df, target_multiclass, "multiclass",
    ["Id", "quality"],
    "Kelompok kualitas: Low <=5, Medium =6, dan High >=7."
)

# MULTILABEL: penanda nilai kimia di atas kuartil ketiga (Q75).
kolom_kimia = {
    "alcohol": "label_high_alcohol",
    "volatile acidity": "label_high_volatile_acidity",
    "sulphates": "label_high_sulphates",
    "residual sugar": "label_high_residual_sugar"
}

target_multilabel = pd.DataFrame(index=df.index)
ambang_wine = {}

for kolom, nama_label in kolom_kimia.items():
    ambang = df[kolom].quantile(0.75)
    ambang_wine[kolom] = round(float(ambang), 4)
    target_multilabel[nama_label] = (df[kolom] >= ambang).astype(int)

hasil_wine_multilabel = simpan_hasil(
    "wine_quality", df, target_multilabel, "multilabel",
    ["Id", "quality"] + list(kolom_kimia.keys()),
    "Empat tag turunan untuk nilai alcohol, volatile acidity, sulphates, dan residual sugar yang berada pada atau di atas Q75."
)

print("Ambang Q75 karakteristik wine:", ambang_wine)

wine_quality | binary | 1143 baris | tersimpan: wine_quality_binary.csv
wine_quality | multiclass | 1143 baris | tersimpan: wine_quality_multiclass.csv
wine_quality | multilabel | 1143 baris | tersimpan: wine_quality_multilabel.csv
Ambang Q75 karakteristik wine: {'alcohol': 11.1, 'volatile acidity': 0.64, 'sulphates': 0.73, 'residual sugar': 2.6}


In [15]:
# 14. VALIDASI DAN MENYIMPAN RINGKASAN HASIL

df_hasil = pd.DataFrame(ringkasan_hasil)
df_panduan = pd.DataFrame(panduan_label)

# Pastikan terdapat 27 file: 9 dataset x 3 jenis klasifikasi.
jumlah_file = sum(
    1 for jenis in ["binary", "multiclass", "multilabel"]
    for file in (OUTPUT_DIR / jenis).glob("*.csv")
)

print("Jumlah dataset sumber:", len(datasets))
print("Jumlah file klasifikasi:", jumlah_file)
print("Total baris dari sembilan dataset:", sum(len(df) for df in datasets.values()))

display(df_hasil[[
    "Dataset", "Jenis Klasifikasi", "Jumlah Baris",
    "Jumlah Fitur", "Kolom Target"
]])

if jumlah_file != 27:
    raise RuntimeError(f"Seharusnya 27 file, tetapi ditemukan {jumlah_file}.")

if len(df_hasil) != 27:
    raise RuntimeError(f"Ringkasan seharusnya berisi 27 baris, tetapi {len(df_hasil)}.")

# Simpan ringkasan dan panduan label.
df_hasil.to_csv(OUTPUT_DIR / "ringkasan_hasil.csv", index=False)
df_panduan.to_csv(OUTPUT_DIR / "panduan_label.csv", index=False)

print("\nValidasi selesai.")
print("Folder hasil:", OUTPUT_DIR)
print("File tambahan: ringkasan_hasil.csv dan panduan_label.csv")

Jumlah dataset sumber: 9
Jumlah file klasifikasi: 27
Total baris dari sembilan dataset: 141571


,Dataset,Jenis Klasifikasi,Jumlah Baris,Jumlah Fitur,Kolom Target
0,heart_failure,binary,299,12,target_binary
1,heart_failure,multiclass,299,11,target_multiclass
2,heart_failure,multilabel,299,8,"label_anaemia, label_diabetes, label_high_blood_pressure, label_smoking"
3,mbg_indonesia,binary,13000,24,target_binary
4,mbg_indonesia,multiclass,13000,24,target_multiclass
5,mbg_indonesia,multilabel,13000,17,"label_high_plate_waste, label_high_spoilage_risk, label_long_delivery_distan..."
6,kepler,binary,9564,42,target_binary
7,kepler,multiclass,9564,42,target_multiclass
8,kepler,multilabel,9564,38,"label_fp_nt, label_fp_ss, label_fp_co, label_fp_ec"
9,mall_customers,binary,200,3,target_binary



Validasi selesai.
Folder hasil: C:\Users\Carxiel\Documents\Tugas Kuliah\Machine Learning\Tugas Binary, Multiclass, Multilabel\hasil_klasifikasi
File tambahan: ringkasan_hasil.csv dan panduan_label.csv


# KESIMPULAN

Berdasarkan proses pengolahan terhadap sembilan dataset, telah dibuat tiga jenis dataset klasifikasi, yaitu **Binary Classification**, **Multiclass Classification**, dan **Multilabel Classification**. Setiap dataset menghasilkan tiga file CSV sehingga terdapat **27 file hasil klasifikasi**.

Binary Classification digunakan untuk membagi data ke dalam dua kelas. Multiclass Classification digunakan ketika target memiliki lebih dari dua kelas. Sementara itu, Multilabel Classification memungkinkan satu baris data memiliki beberapa label sekaligus.

Label pada beberapa dataset menggunakan kolom target asli, sedangkan label lainnya dibentuk melalui aturan tertentu, seperti pengelompokan umur, batas nilai, kuartil, kategori, atau pencarian kata kunci. Oleh karena itu, label turunan merupakan label untuk kebutuhan latihan pengolahan data dan tidak selalu mewakili kategori resmi dari sumber dataset.

Hasil pengolahan disimpan pada folder `hasil_klasifikasi` yang terdiri dari subfolder `binary`, `multiclass`, dan `multilabel`. File `ringkasan_hasil.csv` berisi ringkasan hasil, sedangkan `panduan_label.csv` menjelaskan pembentukan target pada setiap dataset.

**Kesimpulannya**, project ini menunjukkan bahwa satu dataset dapat disiapkan untuk beberapa pendekatan klasifikasi dengan menentukan target yang sesuai, memisahkan kolom sumber target dari fitur, serta memeriksa kembali hasil pengolahan sebelum digunakan pada tahap pemodelan Machine Learning.